# ⚡ Notebook 2: Advanced Agentic Workflows — Deep Research on Your H100

**Workshop Notebook 2 of 2**

In Notebook 1 you saw Hermes running two ways: a 1B model on your own laptop (functional but limited), and Hermes-3-Llama-3.1-8B on an L4 GPU (reliable tool-calling for a moderately complex skill).

This notebook goes further:

1. **Connect Hermes to Qwen2.5-32B-Instruct on your H100 NVL VM** — a genuinely bigger, more capable model that physically cannot fit on the L4's 24GB card, running with a 128K context window.
2. **Give Hermes a real tool**: a research-paper search MCP server, so it searches actual peer-reviewed/academic literature instead of generating an answer from memory.
3. **Run a Deep Research skill** that forces Hermes to decompose a question, call the research tool multiple times, and synthesize a cited answer — genuine multi-step agentic tool use.

### A note on model choice

Two other candidates were tried and dropped for this notebook: **Hermes-4.3-36B** (built on a newer/less mature base architecture) failed to load cleanly on this VM, and **Gemma 4 31B** has open, current vLLM bugs where tool calls or reasoning tags leak into raw output instead of being parsed correctly (verified against a live GitHub issue matching this exact stack). **Qwen2.5-32B-Instruct** is the choice here instead: one of the most battle-tested tool-calling models available (~2 years in production use), with the `hermes` tool-call parser already proven reliable in this workshop. Its one wrinkle — a 32K default context window — is a known, documented config fix (YaRN scaling), not a mystery bug, so it was worth the extra setup step.

---

### Why this genuinely needs the H100

| | Llama 3.2 1B (Laptop, NB1) | Hermes-3-Llama-3.1-8B (L4 GPU, NB1) | **Qwen2.5-32B-Instruct (H100, this notebook)** |
|--|--|--|--|
| Parameters | 1B | 8B | **32B** |
| Weights at FP8 | ~1GB | ~8GB | **~32GB — already exceeds the L4's entire 24GB card** |
| Context window | 128K (capped at 64K for Hermes) | 64K | **128K (YaRN-extended)** |
| Hosting | Your MacBook / Ollama | Your L4 VM / vLLM | **Your H100 NVL VM / vLLM** |
| Tool calling | Unreliable | Reliable (`hermes` parser) | Reliable (`hermes` parser), same proven format |
| Best for | Learning the mechanics | Moderate-complexity skills (spam detection) | **A model and context window the L4 physically cannot run** |

---
> **Prerequisite:** Complete Notebook 1 first — Hermes should already be installed on your machine.
---

## Part 1 — Verify Hermes Is Installed

In [25]:
import subprocess
import os
import sys
import yaml
import socket
import time
from urllib.parse import urlparse

# Ensure hermes is in PATH
home = os.path.expanduser("~")
local_bin = os.path.join(home, ".local", "bin")
if local_bin not in os.environ["PATH"]:
    os.environ["PATH"] = local_bin + ":" + os.environ["PATH"]

# Check hermes version
result = subprocess.run(["hermes", "--version"], capture_output=True, text=True, env=os.environ)
if result.returncode == 0:
    print(f"✅ Hermes found: {result.stdout.strip()}")
else:
    print("❌ Hermes not found. Please run Notebook 1 first to install it.")
    print("   Or install: curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash")

✅ Hermes found: Hermes Agent v0.18.2 (2026.7.7.2) · upstream e4ea0a0e
Install directory: /Users/mohanrajdavala/.hermes/hermes-agent
Install method: git
Python: 3.11.15
OpenAI SDK: 2.24.0
Update available: 1 commit behind — run 'hermes update'


In [26]:
# Light check that Notebook 1 was completed (not a hard requirement — just a heads-up)

hermes_dir = os.path.expanduser("~/.hermes")
skills_dir = os.path.join(hermes_dir, "skills")
config_path = os.path.join(hermes_dir, "config.yaml")

nb1_skills = ["code-reviewer", "spam-detector"]
found = [s for s in nb1_skills if os.path.exists(os.path.join(skills_dir, s, "SKILL.md"))]

if found:
    print(f"✅ Found {len(found)}/{len(nb1_skills)} skills from Notebook 1: {', '.join(found)}")
else:
    print("⚠️  No skills from Notebook 1 found — that's fine, this notebook is self-contained,")
    print("   but you'll get more out of it if you've done Notebook 1 first.")

# Keep old_model around so later cells can report what we switched from
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        _cfg = yaml.safe_load(f) or {}
    print(f"ℹ️  Current Hermes model config: {_cfg.get('model', 'not set')}")
else:
    print("ℹ️  No Hermes config found yet — Part 2 below will create one.")

✅ Found 2/2 skills from Notebook 1: code-reviewer, spam-detector
ℹ️  Current Hermes model config: {'context_length': 65536, 'default': 'hermes-3-llama-3.1-8b', 'provider': 'l4vllm'}


---
## Part 2 — Connect Hermes to Your H100 NVL VM

### Step 1 — Download the model to a plain local directory

Downloading to a flat directory (instead of the symlinked HF cache format) makes it easy to directly edit `config.json` in the next step:

```bash
hf download Qwen/Qwen2.5-32B-Instruct --local-dir /dev/shm/qwen2.5-32b-yarn
```

### Step 2 — Enable YaRN long-context scaling

Qwen2.5-32B-Instruct ships with a 32,768-token default context — below Hermes's 64,000-token floor. Qwen's own docs recommend this exact YaRN patch to extend it to 128K; we use it as-is rather than inventing an untested value:

```bash
python3 -c "
import json
path = '/dev/shm/qwen2.5-32b-yarn/config.json'
with open(path) as f:
    cfg = json.load(f)
cfg['rope_scaling'] = {
    'factor': 4.0,
    'original_max_position_embeddings': 32768,
    'type': 'yarn'
}
with open(path, 'w') as f:
    json.dump(cfg, f, indent=2)
print('rope_scaling added:', cfg['rope_scaling'])
"
grep -A4 rope_scaling /dev/shm/qwen2.5-32b-yarn/config.json
```

We patch the config file directly rather than passing a `--rope-scaling` CLI flag to vLLM — that flag has proven unstable across vLLM versions (present in 0.11.0, silently rejected in 0.11.1). Editing `config.json` is version-independent.

### Step 3 — Serve it

Serving from the local patched directory also sidesteps the HF cache-path bug we hit earlier entirely, since loading from a local path skips the download/cache lookup logic:

```bash
docker run -d --gpus all \
  --name qwen25-32b-vllm-h100 \
  -p 8000:8000 \
  --env TMPDIR=/tmp/hf-staging \
  -v /dev/shm/qwen2.5-32b-yarn:/model \
  -v /dev/shm/hf-tmp:/tmp/hf-staging \
  --ipc=host \
  --restart unless-stopped \
  vllm/vllm-openai:latest \
  --model /model \
  --served-model-name Qwen2.5-32B-Instruct \
  --quantization fp8 \
  --dtype auto \
  --gpu-memory-utilization 0.90 \
  --max-model-len 131072 \
  --enable-auto-tool-choice \
  --tool-call-parser hermes \
  --api-key "111831022a2f7e5a72e18aaed19dbf22c6d46998797b82255c682bb24613382f" \
  --host 0.0.0.0 \
  --port 8000
```

### Why this is the genuine "why H100" story

- **32B at FP8 ≈ 32GB of weights alone** — that already doesn't fit in the L4's 24GB card, before counting any KV cache. This is a hard capability ceiling the L4 cannot cross, not a convenience choice.
- **128K context** (`--max-model-len 131072`) — double Hermes's 64K floor, using Qwen's own tested YaRN configuration.
- **`--tool-call-parser hermes`** — the exact same proven parser already validated on the L4 in Notebook 1. Qwen2.5's chat template natively uses the Hermes-style `<tool_call>` format, so this is a well-documented, mature combination — not a guess.
- **VRAM headroom:** ~32GB weights against an ~84.6GB pool (94GB × 0.90 utilization) leaves ~52.6GB for KV cache at 128K context — comfortable margin for this workshop's sequential single-request demo.

### Fill in your H100 endpoint details
Once the container is up and the port is forwarded, fill these in below (same pattern as the L4 GPU in Notebook 1).

In [27]:
# ============================================================
# 🔑 H100 VM ENDPOINT DETAILS
# ============================================================

H100_BASE_URL = "http://103.42.51.73:1028/v1"
H100_MODEL    = "Qwen2.5-32B-Instruct"
H100_API_KEY  = "111831022a2f7e5a72e18aaed19dbf22c6d46998797b82255c682bb24613382f"

# ============================================================

if "103.42.51.73" in H100_BASE_URL:
    print(f"✅ H100 endpoint set: {H100_BASE_URL}")
    print(f"✅ Model: {H100_MODEL}")
else:
    print("❌ Please fill in H100_BASE_URL above with your VM's public IP and forwarded port")
    print("   (Same process as the L4 GPU port-forward setup in Notebook 1, Part 5a)")

✅ H100 endpoint set: http://103.42.51.73:1028/v1
✅ Model: Qwen2.5-32B-Instruct


In [28]:
# Quick TCP connectivity check before touching Hermes config
# (Catches port-forwarding / firewall problems early, same as the L4 GPU check in Notebook 1)

if "103.42.51.73" in H100_BASE_URL:
    parsed = urlparse(H100_BASE_URL)
    host, port = parsed.hostname, parsed.port or 80
    print(f"Testing TCP connection to {host}:{port} ...")
    try:
        sock = socket.create_connection((host, port), timeout=8)
        sock.close()
        print(f"✅ Reachable: {host}:{port}")
    except Exception as ex:
        print(f"❌ Could not reach {host}:{port} — {ex}")
        print("   Check: is the container running? Is the port forwarded? Any firewall/security-group rule needed?")
else:
    print("⏭️  Skipping — fill in H100_BASE_URL in the cell above first")

Testing TCP connection to 103.42.51.73:1028 ...
✅ Reachable: 103.42.51.73:1028


In [29]:
# Reconfigure Hermes to point at your H100 VM
#
# NOTE: this uses a NAMED provider entry under "providers:", not the inline
# "model: {provider: custom, base_url: ...}" shorthand. That inline path has a
# confirmed, open Hermes Agent bug (github.com/NousResearch/hermes-agent/issues/43586):
# it never reads api_key or key_env from the model block at all, and silently
# sends "Authorization: Bearer no-key-required" instead - which is exactly the
# HTTP 401 we hit. The named-provider path does not have this bug.

config_path = os.path.expanduser("~/.hermes/config.yaml")
env_path = os.path.expanduser("~/.hermes/.env")

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
else:
    config = {}

old_model = config.get("model", "unknown")

PROVIDER_NAME = "h100vllm"

config["model"] = {
    "default": H100_MODEL,
    "provider": PROVIDER_NAME,
}

if "providers" not in config:
    config["providers"] = {}

config["providers"][PROVIDER_NAME] = {
    "name": PROVIDER_NAME,
    "base_url": H100_BASE_URL,
    "key_env": "H100_API_KEY",
    "default_model": H100_MODEL,
    "api_mode": "chat_completions",
    # Ask vLLM to emit a final usage chunk on streamed responses so Hermes can
    # record real input/output token counts. Without this, Hermes's own
    # streaming-usage handling silently records 0 tokens for providers that
    # don't return `usage` in the stream - a confirmed, known Hermes Agent
    # limitation (github.com/NousResearch/hermes-agent issues #31559, #12023).
    # vLLM's OpenAI-compatible server does honor this flag, so this should
    # give us real counts; the metrics helper below still estimates tokens
    # from the transcript as a safety net if the count comes back 0 anyway.
    "extra_body": {
        "stream_options": {"include_usage": True},
    },
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

# key_env points at this variable name - it must exist in .env
env_vars = {}
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                env_vars[k.strip()] = v.strip()

env_vars["H100_API_KEY"] = H100_API_KEY

with open(env_path, "w") as f:
    for k, v in env_vars.items():
        f.write(f"{k}={v}\n")

print(f"✅ Hermes reconfigured!")
print(f"   Before: {old_model}")
print(f"   After:  {H100_MODEL} via provider '{PROVIDER_NAME}' -> your H100 VM")

# Diagnostic: show exactly what Hermes itself thinks is configured right now.
diag = subprocess.run(["hermes", "config", "show"], capture_output=True, text=True, env=os.environ)
print("\n--- hermes config show (model/providers section) ---")
printing = False
for line in (diag.stdout or diag.stderr).splitlines():
    low = line.strip().lower()
    if low.startswith("model") or low.startswith("providers") or "base_url" in low or "key_env" in low or "api_key" in low:
        print(line)

✅ Hermes reconfigured!
   Before: {'context_length': 65536, 'default': 'hermes-3-llama-3.1-8b', 'provider': 'l4vllm'}
   After:  Qwen2.5-32B-Instruct via provider 'h100vllm' -> your H100 VM

--- hermes config show (model/providers section) ---
  Model:        {'default': 'Qwen2.5-32B-Instruct', 'provider': 'h100vllm'}
  Model:        (auto)


In [30]:
# Verify the connection to your H100 VM

print(f"Testing connection to {H100_MODEL} on your H100 VM...")

result = subprocess.run(
    ["hermes", "-z", "Say exactly: 'H100 GPU connection successful!' and nothing else."],
    capture_output=True, text=True, timeout=60, env=os.environ
)

response = result.stdout or result.stderr
print("Response:", response.strip())

if "successful" in response.lower():
    print(f"\n🚀 Connected to your H100 VM with {H100_MODEL}!")
else:
    print("\n⚠️  Unexpected response — check H100_BASE_URL, H100_API_KEY, and that the container is healthy")

Testing connection to Qwen2.5-32B-Instruct on your H100 VM...
Response: H100 GPU connection successful!

🚀 Connected to your H100 VM with Qwen2.5-32B-Instruct!


---
## Part 3 — Give Hermes a Real Research Tool (MCP)

An LLM answering research questions from memory alone will confidently make things up. The fix is to give Hermes an actual tool it can call to search real papers — that's what MCP (Model Context Protocol) servers are for.

### What we checked

**Consensus MCP** (`https://mcp.consensus.app/mcp`) does have a genuinely free tier — no account needed, 3 papers per search, unlimited searches per month (a paid account raises that to 10-20 papers/search). The catch: it's a **remote HTTP server that authenticates via OAuth**, designed for interactive clients like Claude Desktop, Cursor, or ChatGPT where a browser pops up to log in. Hermes is a scripted CLI running headless inside a notebook — whether its MCP client supports that OAuth handshake at all is unconfirmed, so it's a poor foundation to build the main demo on.

**`paper-search-mcp`** ([GitHub](https://github.com/openags/paper-search-mcp), MIT license) is the reliable choice instead: completely free, **no API key or account required at all**, runs locally as a standard command-line MCP server (the same `command`/`args` config style Hermes already uses), and searches across arXiv, PubMed, bioRxiv, Semantic Scholar, CORE, OpenAlex, Crossref, and more in one call.

**Plan:** `paper-search-mcp` is the primary tool for this workshop (Part 3a — do this). Consensus is included as an optional bonus (Part 3b) — try it if time permits, but don't depend on it for the live demo.

### Part 3a — Install and configure paper-search-mcp (primary, do this)

In [31]:
# Install paper-search-mcp (free, no API key required)

print("Installing paper-search-mcp...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "paper-search-mcp"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("✅ paper-search-mcp installed")
else:
    print("❌ Install failed:")
    print(result.stderr[-1000:])
    print("\nIf you see an 'externally-managed-environment' error, retry with:")
    print(f'   {sys.executable} -m pip install -q --break-system-packages paper-search-mcp')

Installing paper-search-mcp...
✅ paper-search-mcp installed


In [32]:
# Register paper-search-mcp as an MCP server in Hermes config
# (No API keys required — arXiv, PubMed, bioRxiv, Semantic Scholar, etc. all work key-free.
#  Optional keys below only improve rate limits on a couple of sources - leave blank to skip.)

with open(config_path, "r") as f:
    config = yaml.safe_load(f) or {}

if "mcp_servers" not in config:
    config["mcp_servers"] = {}

config["mcp_servers"]["paper-search"] = {
    "command": sys.executable,
    "args": ["-m", "paper_search_mcp.server"],
    "env": {
        "PAPER_SEARCH_MCP_UNPAYWALL_EMAIL": "",       # optional: your email, enables Unpaywall DOI lookups
        "PAPER_SEARCH_MCP_CORE_API_KEY": "",           # optional: free key from core.ac.uk raises CORE rate limits
        "PAPER_SEARCH_MCP_SEMANTIC_SCHOLAR_API_KEY": "",  # optional: free key raises Semantic Scholar rate limits
    }
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ paper-search-mcp registered with Hermes")
print("   Main tool Hermes will call: search_papers (searches arXiv, PubMed, bioRxiv, Semantic Scholar, CORE, OpenAlex, Crossref, and more)")

✅ paper-search-mcp registered with Hermes
   Main tool Hermes will call: search_papers (searches arXiv, PubMed, bioRxiv, Semantic Scholar, CORE, OpenAlex, Crossref, and more)


In [ ]:
# ============================================================
# ⚠️ OPTIONAL / BONUS — Consensus MCP (free tier, OAuth, unverified with Hermes CLI)
# ============================================================
# This cell is OFF by default. paper-search-mcp above is the reliable primary tool - only turn this on to experiment
#
# The config keys below (`url` / `transport`) are my best interpretation of how a
# remote streamable-HTTP MCP server would be declared in Hermes's config.yaml, based
# on Consensus's docs - I could not find Hermes-specific documentation confirming this
# schema, and the OAuth login step in particular may not work from a headless CLI.
# Test it in isolation before trusting it in a live demo.

ENABLE_CONSENSUS_BONUS = False

if ENABLE_CONSENSUS_BONUS:
    with open(config_path, "r") as f:
        config = yaml.safe_load(f) or {}
    if "mcp_servers" not in config:
        config["mcp_servers"] = {}
    config["mcp_servers"]["consensus"] = {
        "url": "https://mcp.consensus.app/mcp",
        "transport": "http",
    }
    with open(config_path, "w") as f:
        yaml.dump(config, f, default_flow_style=False)
    print("✅ Consensus MCP added (unverified - test with a simple research prompt)")
    print("   Free tier: no account needed, 3 papers/search, unlimited searches/month")
else:
    print("⏭️  Skipped — set ENABLE_CONSENSUS_BONUS = True above to try it")

---
## Part 4 — Create the Deep Research Skill

In [33]:
# Create the deep-research skill
# Rewritten again after a second real test: the model still skipped decomposing
# the question into sub-questions (made only ONE search call) and still tried to
# end its turn with a follow-up question instead of a report. The skill's prose
# alone wasn't strong enough - the CRITICAL block below is deliberately blunt and
# repeats the two failure modes we actually observed, since research shows models
# comply with format/stopping constraints far more reliably when the constraint
# is stated up front, in imperative language, with the actual failure named.

skills_dir = os.path.expanduser("~/.hermes/skills")
os.makedirs(skills_dir, exist_ok=True)

research_skill_content = """---
name: deep-research
description: Answer questions using deep research from real peer-reviewed academic papers via MCP research tools
---

# Deep Research Skill

## CRITICAL - READ THIS FIRST

This is a SINGLE non-interactive turn. There is no user available to answer
follow-up questions. You MUST NOT end your response by asking a question,
offering options, or asking permission to proceed (e.g. "Would you like me to
download this paper?"). If you do this, the task has FAILED. You must instead
use whatever evidence you have gathered and produce the complete report below
as your one and only final message.

You MUST call mcp__paper_search__search_papers (or a source-specific search
tool) at least TWICE, for at least two different sub-questions or search
angles, before writing your report. A single search call is not sufficient
research and the task has FAILED if you only search once.

You are a research assistant with access to academic paper search tools
(mcp__paper_search__search_papers and its per-source variants, covering arXiv,
PubMed, bioRxiv, Semantic Scholar, CORE, OpenAlex, Crossref, PMC, OpenAIRE and
more). You answer questions ONLY based on what the scientific literature says.

## Research Process

1. Decompose the question into 2-4 specific searchable sub-questions.
2. For EACH sub-question, call mcp__paper_search__search_papers once (twice at
   most if the first search returns nothing useful).
3. Abstracts returned by search are SUFFICIENT evidence for this report.
   Downloading/reading full text is optional, best-effort only - many papers
   are paywalled and will fail to download. If a download or read fails, move
   on immediately. Never retry a download more than once per paper, and never
   let a failed download block you from finishing the report.
4. Search for contradicting evidence too, not just confirming evidence.
5. Synthesize findings across multiple papers using their titles, authors, and
   abstracts.
6. Write the full report in the Output Format below as your final message.

## Output Format (produce ALL SIX sections, in this order, every time)

### Question
[Restate clearly]

### Research Searches Performed
1. Tool + query: [tool name, query] -> [results found]
2. Tool + query: [tool name, query] -> [results found]
...(one line per search call you made - there must be at least 2)

### Key Findings
[Synthesis with inline citations, drawn from abstracts. Write actual prose
paragraphs summarizing what the evidence shows - not just a list of paper
titles.]

### Evidence Consensus
[Strong consensus / Mixed evidence / Emerging evidence / Insufficient data]

### Sources
[All papers cited: Author(s), Year, Title, source/journal, URL if available]

### Caveats
[Limitations, contradictions, gaps - including any papers whose full text
could not be retrieved]

## Rules
- NEVER answer without calling a research tool first.
- NEVER end your response with a question back to the user. Ever.
- ALWAYS make at least 2 search calls across different sub-questions.
- ALWAYS produce the full Output Format above, with all six headers, as your
  final message.
- Do not make more than ~6 tool calls total per question.
- Report what the literature says, not personal opinion.
"""

skill_dir = os.path.join(skills_dir, "deep-research")
os.makedirs(skill_dir, exist_ok=True)
with open(os.path.join(skill_dir, "SKILL.md"), "w") as f:
    f.write(research_skill_content)

print("✅ Skill ready: /deep-research")

✅ Skill ready: /deep-research


In [34]:
# Diagnose whether paper-search-mcp is actually connected before trusting any
# research output. If this doesn't show it as connected, Hermes will silently
# fall back to its own generic web_search tool instead - which is what produced
# the "technical page... automated request detected" response.

print("--- hermes mcp list ---")
r1 = subprocess.run(["hermes", "mcp", "list"], capture_output=True, text=True, env=os.environ, timeout=30)
print(r1.stdout or r1.stderr)

print("\n--- hermes mcp test paper-search ---")
r2 = subprocess.run(["hermes", "mcp", "test", "paper-search"], capture_output=True, text=True, env=os.environ, timeout=60)
print(r2.stdout or r2.stderr)

print("\n--- Ask the model directly what tools it can see ---")
r3 = subprocess.run(
    ["hermes", "chat", "-q", "List every tool you currently have access to, exactly as their function/tool names appear to you. Do not call any of them, just list names.", "-v"],
    capture_output=True, text=True, env=os.environ, timeout=60
)
print(r3.stdout or r3.stderr)

--- hermes mcp list ---

  MCP Servers:

  Name             Transport                      Tools        Status    
  ──────────────── ────────────────────────────── ──────────── ──────────
  paper-search     /Users/mohanrajdavala/.ju...   all          ✓ enabled



--- hermes mcp test paper-search ---

  Testing 'paper-search'...
  Transport: stdio → /Users/mohanrajdavala/.jupyter-venv/bin/python
  Auth: none
  ✓ Connected (3120ms)
  ✓ Tools discovered: 57

    search_papers                        Unified top-level search across all configured academic...
    search_arxiv                         Search academic papers from arXiv.

    Args:
        q...
    search_pubmed                        Search academic papers from PubMed.

    Args:
        ...
    search_biorxiv                       Search academic papers from bioRxiv.

    Note: bioRxiv...
    search_medrxiv                       Search academic papers from medRxiv.

    Note: medRxiv...
    search_google_scholar              

In [35]:
# Helper functions to pull real usage metrics for a Hermes session.
#
# Confirmed against the Hermes Agent source (hermes_state.py, SessionDB.export_session):
#   def export_session(self, session_id):
#       session = self.get_session(session_id)   # raw `sessions` table row
#       messages = self.get_messages(session_id)
#       return {**session, "messages": messages}
#
# The exported JSON is a FLAT dict: the `sessions` SQLite row's columns are
# spread at the top level (input_tokens, output_tokens, reasoning_tokens,
# tool_call_count, message_count, api_call_count, estimated_cost_usd, ...),
# plus a "messages" key holding the full per-message transcript.
#
# KNOWN ISSUE - why tokens can still read 0/None even with the fix above:
# Hermes only records real token counts when the model backend returns a
# `usage` object on the final streamed chunk. This is a confirmed, open
# Hermes Agent limitation for some custom/self-hosted OpenAI-compatible
# backends - see github.com/NousResearch/hermes-agent issues #31559 and
# #12023. When that happens, Hermes silently records input_tokens=0 /
# output_tokens=0 with no warning. We asked vLLM for `stream_options:
# {include_usage: true}` in the H100 provider config (previous cell), which
# should fix it - but as a safety net, if the DB still shows 0/0 despite a
# non-trivial transcript, we fall back to a rough character-based token
# estimate (~4 chars/token) computed directly from the exported message
# content, and clearly flag those numbers as ESTIMATED rather than exact.

import re, json as _json, tempfile

def _rough_token_estimate(value):
    """~4 chars/token heuristic estimate for English text. Accepts str or
    JSON-serializable structures (e.g. tool_calls lists)."""
    if not value:
        return 0
    text = value if isinstance(value, str) else _json.dumps(value)
    return max(1, len(text) // 4)


def get_hermes_metrics(hermes_output):
    """Parse a Hermes CLI run's stdout/stderr, pull its Session ID, then
    export that session from ~/.hermes/state.db to get real token/tool-call
    counts (the CLI's own live-run summary does not print token counts)."""
    m = re.search(r"Session:\s+(\S+)", hermes_output)
    if not m:
        return {"error": "Could not find a Session ID in the Hermes output "
                          "(expected a line like 'Session:  20260714_...')."}
    session_id = m.group(1)

    with tempfile.NamedTemporaryFile(mode="w", suffix=".jsonl", delete=False) as tf:
        export_path = tf.name

    export = subprocess.run(
        ["hermes", "sessions", "export", export_path, "--session-id", session_id],
        capture_output=True, text=True, env=os.environ, timeout=30
    )

    metrics = {"session_id": session_id}
    try:
        with open(export_path) as f:
            lines = [l for l in f if l.strip()]
        if not lines:
            metrics["error"] = f"Export produced no data. hermes stderr: {export.stderr.strip()[:300]}"
            return metrics

        data = _json.loads(lines[0])

        # Session-level fields (flat, from the `sessions` table row)
        metrics["model"] = data.get("model")
        metrics["message_count"] = data.get("message_count")
        metrics["tool_call_count"] = data.get("tool_call_count")
        metrics["api_call_count"] = data.get("api_call_count")
        metrics["input_tokens"] = data.get("input_tokens", 0) or 0
        metrics["output_tokens"] = data.get("output_tokens", 0) or 0
        metrics["reasoning_tokens"] = data.get("reasoning_tokens", 0) or 0
        metrics["cache_read_tokens"] = data.get("cache_read_tokens", 0) or 0
        metrics["cache_write_tokens"] = data.get("cache_write_tokens", 0) or 0
        metrics["estimated_cost_usd"] = data.get("estimated_cost_usd")

        messages = data.get("messages", [])

        # Cross-check: derive tool names + a fallback tool-call count directly
        # from the message transcript, in case the session-level counter is
        # ever absent on an older DB schema.
        tool_names = set()
        derived_tool_call_count = 0
        for msg in messages:
            tc_list = msg.get("tool_calls") or []
            if isinstance(tc_list, list):
                for tc in tc_list:
                    derived_tool_call_count += 1
                    name = (tc.get("function") or {}).get("name") or tc.get("name")
                    if name:
                        tool_names.add(name)
            if msg.get("role") == "tool" and msg.get("tool_name"):
                tool_names.add(msg["tool_name"])
        metrics["unique_tools_used"] = sorted(tool_names)
        if metrics["tool_call_count"] is None:
            metrics["tool_call_count"] = derived_tool_call_count

        # Token fallback: if the backend never returned real usage data, the
        # DB legitimately has 0/0 even though real work happened. Estimate
        # from message content instead of showing a misleading blank.
        metrics["tokens_are_estimated"] = False
        if not metrics["input_tokens"] and not metrics["output_tokens"] and messages:
            est_in, est_out = 0, 0
            for msg in messages:
                role = msg.get("role")
                tok = _rough_token_estimate(msg.get("content"))
                tok += _rough_token_estimate(msg.get("tool_calls"))
                if role == "assistant":
                    est_out += tok
                else:
                    est_in += tok
            metrics["input_tokens"] = est_in
            metrics["output_tokens"] = est_out
            metrics["tokens_are_estimated"] = True

        metrics["_raw_keys"] = sorted(data.keys())
    except Exception as ex:
        metrics["error"] = f"Could not parse export: {ex}"
    finally:
        try:
            os.remove(export_path)
        except OSError:
            pass
    return metrics


def print_metrics(metrics, label):
    print(f"\n📊 === METRICS: {label} ===")
    if metrics.get("error"):
        print(f"   ⚠️  {metrics['error']}")
        if "_raw_keys" in metrics:
            print(f"   (Available fields in export: {metrics['_raw_keys']})")
        return
    print(f"   Session ID:          {metrics.get('session_id')}")
    if metrics.get("model"):
        print(f"   Model:               {metrics['model']}")
    print(f"   Messages (turns):    {metrics.get('message_count', 'n/a')}")
    print(f"   API calls to LLM:    {metrics.get('api_call_count', 'n/a')}")
    print(f"   Tool calls made:     {metrics.get('tool_call_count', 'n/a')}")
    if metrics.get("unique_tools_used"):
        print(f"   Tools used:          {', '.join(metrics['unique_tools_used'])}")
    in_tok = metrics.get("input_tokens")
    out_tok = metrics.get("output_tokens")
    tag = "  (estimated - backend returned no usage data)" if metrics.get("tokens_are_estimated") else ""
    print(f"   Input tokens:        {in_tok if in_tok is not None else 'n/a'}{tag}")
    print(f"   Output tokens:       {out_tok if out_tok is not None else 'n/a'}{tag}")
    if metrics.get("reasoning_tokens"):
        print(f"   Reasoning tokens:    {metrics['reasoning_tokens']}")
    if isinstance(in_tok, int) and isinstance(out_tok, int):
        print(f"   Total tokens:        {in_tok + out_tok}")
    if metrics.get("estimated_cost_usd") is not None:
        print(f"   Estimated cost:      ${metrics['estimated_cost_usd']:.6f}")


---
## Part 5 — Deep Research in Action

Three research questions, each requiring Hermes to decompose, search, and synthesize across multiple real papers.

In [36]:
research_q1 = """/deep-research 
What does the scientific research say about the effectiveness of intermittent fasting 
for weight loss compared to continuous caloric restriction? 
Only answer based on peer-reviewed studies.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

print(f"Researching Question 1 with {H100_MODEL}...")
print("=" * 60)

result = subprocess.run(
    ["hermes", "chat", "-q", research_q1, "-v", "--toolsets", "paper-search,skills"],
    capture_output=True, text=True, timeout=300, env=os.environ
)

print(f"=== RESEARCH RESULTS ({H100_MODEL}) ===")
print(result.stdout or result.stderr)

Researching Question 1 with Qwen2.5-32B-Instruct...
=== RESEARCH RESULTS (Qwen2.5-32B-Instruct) ===
Query: /deep-research 
What does the scientific research say about the effectiveness of intermittent 
fasting 
for weight loss compared to continuous caloric restriction? 
Only answer based on peer-reviewed studies.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found.
Initializing agent...
────────────────────────────────────────
🤖 AI Agent initialized with model: Qwen2.5-32B-Instruct
🔗 Using custom base URL: http://103.42.51.73:1028/v1
🔑 Using API key: 11183

In [37]:
# Metrics for Question 1
metrics_q1 = get_hermes_metrics(result.stdout or result.stderr)
print_metrics(metrics_q1, "Question 1 - Intermittent Fasting")


📊 === METRICS: Question 1 - Intermittent Fasting ===
   Session ID:          20260715_012159_34b356
   Model:               Qwen2.5-32B-Instruct
   Messages (turns):    5
   API calls to LLM:    2
   Tool calls made:     2
   Tools used:          mcp__paper_search__search_papers
   Input tokens:        52452
   Output tokens:       1463
   Total tokens:        53915
   Estimated cost:      $0.000000


In [38]:
research_q2 = """/deep-research
What are the neurobiological mechanisms by which sleep deprivation impairs cognitive 
performance, and what does the evidence say about recovery — does one night of 
recovery sleep fully restore baseline performance?
Provide a comprehensive, citation-backed answer.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

print(f"Researching Question 2 (multi-hop) with {H100_MODEL}...")
print("=" * 60)

result = subprocess.run(
    ["hermes", "chat", "-q", research_q2, "-v", "--toolsets", "paper-search,skills"],
    capture_output=True, text=True, timeout=360, env=os.environ
)

print(f"=== RESEARCH RESULTS ({H100_MODEL}) ===")
print(result.stdout or result.stderr)

Researching Question 2 (multi-hop) with Qwen2.5-32B-Instruct...
=== RESEARCH RESULTS (Qwen2.5-32B-Instruct) ===
Query: /deep-research
What are the neurobiological mechanisms by which sleep deprivation impairs 
cognitive 
performance, and what does the evidence say about recovery — does one night of 
recovery sleep fully restore baseline performance?
Provide a comprehensive, citation-backed answer.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found.
Initializing agent...
────────────────────────────────────────
🤖 AI Agent initialized with model: Qwen2.5-32B

In [39]:
# Metrics for Question 2
metrics_q2 = get_hermes_metrics(result.stdout or result.stderr)
print_metrics(metrics_q2, "Question 2 - Sleep Deprivation")


📊 === METRICS: Question 2 - Sleep Deprivation ===
   Session ID:          20260715_012258_981454
   Model:               Qwen2.5-32B-Instruct
   Messages (turns):    5
   API calls to LLM:    2
   Tool calls made:     2
   Tools used:          mcp__paper_search__search_papers
   Input tokens:        59769
   Output tokens:       1308
   Total tokens:        61077
   Estimated cost:      $0.000000


In [40]:
research_q3 = """/deep-research
Is there scientific evidence that omega-3 fatty acid supplementation reduces the risk of 
cardiovascular disease? If so, what dosage does the evidence support, and does it matter 
whether the source is fish oil vs algae-based omega-3? Synthesize the most recent evidence.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

print(f"Researching Question 3 (complex, multi-part) with {H100_MODEL}...")
print("=" * 60)

result = subprocess.run(
    ["hermes", "chat", "-q", research_q3, "-v", "--toolsets", "paper-search,skills"],
    capture_output=True, text=True, timeout=360, env=os.environ
)

print(f"=== RESEARCH RESULTS ({H100_MODEL}) ===")
print(result.stdout or result.stderr)

Researching Question 3 (complex, multi-part) with Qwen2.5-32B-Instruct...
=== RESEARCH RESULTS (Qwen2.5-32B-Instruct) ===
Query: /deep-research
Is there scientific evidence that omega-3 fatty acid supplementation reduces the
risk of 
cardiovascular disease? If so, what dosage does the evidence support, and does 
it matter 
whether the source is fish oil vs algae-based omega-3? Synthesize the most 
recent evidence.

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found.
Initializing agent...
────────────────────────────────────────
🤖 AI Agent initialized with m

In [41]:
# Metrics for Question 3
metrics_q3 = get_hermes_metrics(result.stdout or result.stderr)
print_metrics(metrics_q3, "Question 3 - Omega-3")


📊 === METRICS: Question 3 - Omega-3 ===
   Session ID:          20260715_012357_c704ff
   Model:               Qwen2.5-32B-Instruct
   Messages (turns):    6
   API calls to LLM:    2
   Tool calls made:     3
   Tools used:          mcp__paper_search__search_papers
   Input tokens:        72390
   Output tokens:       1344
   Total tokens:        73734
   Estimated cost:      $0.000000


In [42]:
print("📊 === SUMMARY: TOKEN & TOOL-CALL USAGE (SEQUENTIAL Q1-Q3) ===\n")

all_metrics = [
    ("Q1 Intermittent Fasting", metrics_q1),
    ("Q2 Sleep Deprivation", metrics_q2),
    ("Q3 Omega-3", metrics_q3),
]

seq_total_in, seq_total_out, seq_total_calls, seq_total_api_calls = 0, 0, 0, 0
for label, m in all_metrics:
    if m.get("error"):
        print(f"{label}: unavailable ({m['error']})")
        continue
    in_tok = m.get("input_tokens") or 0
    out_tok = m.get("output_tokens") or 0
    calls = m.get("tool_call_count") or 0
    api_calls = m.get("api_call_count") or 0
    seq_total_in += in_tok
    seq_total_out += out_tok
    seq_total_calls += calls
    seq_total_api_calls += api_calls
    print(f"{label:28s} | tool calls: {calls:3d} | LLM API calls: {api_calls:2d} | input tok: {in_tok:7d} | output tok: {out_tok:6d}")

print(f"\n{'SEQUENTIAL TOTAL':28s} | tool calls: {seq_total_calls:3d} | LLM API calls: {seq_total_api_calls:2d} | input tok: {seq_total_in:7d} | output tok: {seq_total_out:6d}")
print(f"{'':28s} | grand total tokens: {seq_total_in + seq_total_out}")


📊 === SUMMARY: TOKEN & TOOL-CALL USAGE (SEQUENTIAL Q1-Q3) ===

Q1 Intermittent Fasting      | tool calls:   2 | LLM API calls:  2 | input tok:   52452 | output tok:   1463
Q2 Sleep Deprivation         | tool calls:   2 | LLM API calls:  2 | input tok:   59769 | output tok:   1308
Q3 Omega-3                   | tool calls:   3 | LLM API calls:  2 | input tok:   72390 | output tok:   1344

SEQUENTIAL TOTAL             | tool calls:   7 | LLM API calls:  6 | input tok:  184611 | output tok:   4115
                             | grand total tokens: 188726


---
## Part 5b — Parallel Load Test: 5 Simultaneous Requests to H100

So far every question has been sent to Hermes **one at a time, waiting for
each to finish before starting the next**. That proves the pipeline works,
but it doesn't prove the H100 is actually being used efficiently.

vLLM serves models with **continuous batching**: it can process multiple
requests' token generation steps together on the same GPU, instead of
running one request start-to-finish before touching the next. This section
fires **5 different research questions at Hermes simultaneously** (via
Python threads, each spawning its own `hermes chat` process) and measures:

- Whether the H100/vLLM endpoint actually serves them concurrently (wall-clock
  time for the batch vs. the sum of each request's own duration)
- Total tool calls, LLM API calls, and tokens consumed across all 5 requests
- Per-question metrics, exactly like the sequential run above

Five new topics, same `/deep-research` skill and reinforcement prompt pattern
as Q1-Q3.

In [43]:
# Five new research questions - these will all be sent to Hermes / H100
# AT THE SAME TIME in the next cell, instead of one after another.

research_q4 = """/deep-research
Does creatine monohydrate supplementation increase muscle strength and lean
body mass in resistance-trained adults, according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

research_q5 = """/deep-research
Does vitamin D supplementation reduce the risk or severity of respiratory
tract infections, according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

research_q6 = """/deep-research
Does mindfulness meditation measurably lower cortisol and other physiological
stress biomarkers, according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

research_q7 = """/deep-research
For fat loss, is high-intensity interval training (HIIT) more effective than
steady-state cardio, according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

research_q8 = """/deep-research
Do probiotic supplements meaningfully reduce symptoms of irritable bowel
syndrome (IBS), according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found."""

print("Defined 5 parallel research questions (Q4-Q8).")

Defined 5 parallel research questions (Q4-Q8).


In [44]:
import concurrent.futures
import time

def run_hermes_question(question_prompt):
    """Run one hermes chat call and time it. Executed inside a worker thread."""
    start = time.time()
    result = subprocess.run(
        ["hermes", "chat", "-q", question_prompt, "-v", "--toolsets", "paper-search,skills"],
        capture_output=True, text=True, timeout=360, env=os.environ
    )
    elapsed = time.time() - start
    return {"result": result, "elapsed": elapsed}

parallel_questions = [
    ("Q4 Creatine & Muscle Strength", research_q4),
    ("Q5 Vitamin D & Respiratory Infections", research_q5),
    ("Q6 Meditation & Cortisol", research_q6),
    ("Q7 HIIT vs Steady-State Cardio", research_q7),
    ("Q8 Probiotics & IBS", research_q8),
]

print(f"\U0001F680 Firing {len(parallel_questions)} requests at H100 SIMULTANEOUSLY...")
print(f"   Model: {H100_MODEL}  |  Endpoint: {H100_BASE_URL}")
print("   Each request is a separate Hermes CLI process running in its own thread.")
print("   This tests whether vLLM's continuous batching serves them concurrently")
print("   instead of silently queueing them one at a time.")
print("=" * 70)

batch_start = time.time()
parallel_results = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=len(parallel_questions)) as executor:
    future_to_label = {
        executor.submit(run_hermes_question, prompt): label
        for label, prompt in parallel_questions
    }
    for future in concurrent.futures.as_completed(future_to_label):
        label = future_to_label[future]
        try:
            outcome = future.result()
            parallel_results[label] = outcome
            print(f"\u2705 {label:38s} finished in {outcome['elapsed']:6.1f}s")
        except Exception as exc:
            parallel_results[label] = {"error": str(exc), "elapsed": None}
            print(f"\u274c {label:38s} failed: {exc}")

batch_elapsed = time.time() - batch_start
print("=" * 70)
print(f"\U0001F3C1 All {len(parallel_questions)} parallel requests completed in {batch_elapsed:.1f}s wall-clock")

🚀 Firing 5 requests at H100 SIMULTANEOUSLY...
   Model: Qwen2.5-32B-Instruct  |  Endpoint: http://103.42.51.73:1028/v1
   Each request is a separate Hermes CLI process running in its own thread.
   This tests whether vLLM's continuous batching serves them concurrently
   instead of silently queueing them one at a time.
✅ Q6 Meditation & Cortisol               finished in   75.3s
✅ Q5 Vitamin D & Respiratory Infections  finished in   75.5s
✅ Q4 Creatine & Muscle Strength          finished in   78.5s
✅ Q7 HIIT vs Steady-State Cardio         finished in   79.0s
✅ Q8 Probiotics & IBS                    finished in   80.0s
🏁 All 5 parallel requests completed in 80.1s wall-clock


In [45]:
# Print each transcript and pull real per-question metrics, reusing the same
# get_hermes_metrics()/print_metrics() helpers from the sequential section.

parallel_metrics = {}
for label, prompt in parallel_questions:
    outcome = parallel_results.get(label, {})
    print(f"\n{'=' * 70}\n=== RESEARCH RESULTS: {label} ===\n{'=' * 70}")
    if outcome.get("elapsed") is None:
        print(f"\u274c Request failed: {outcome.get('error')}")
        parallel_metrics[label] = {"error": outcome.get("error", "unknown failure")}
        continue
    output_text = outcome["result"].stdout or outcome["result"].stderr
    print(output_text)
    m = get_hermes_metrics(output_text)
    parallel_metrics[label] = m
    print_metrics(m, label)


=== RESEARCH RESULTS: Q4 Creatine & Muscle Strength ===
Query: /deep-research
Does creatine monohydrate supplementation increase muscle strength and lean
body mass in resistance-trained adults, according to peer-reviewed studies?

IMPORTANT: This is a single non-interactive turn - there is no user available
to answer follow-up questions. Decompose this into at least 2 sub-questions,
call the paper search tool separately for each, and then output the complete
six-section report (Question / Research Searches Performed / Key Findings /
Evidence Consensus / Sources / Caveats) as your final message. Do not end by
asking a question or offering options - deliver the full report now, using
whatever evidence you found.
Initializing agent...
────────────────────────────────────────
🤖 AI Agent initialized with model: Qwen2.5-32B-Instruct
🔗 Using custom base URL: http://103.42.51.73:1028/v1
🔑 Using API key: 11183102...382f
✅ Enabled toolset 'paper-search': mcp__paper_search__download_arxiv, mcp__

In [46]:
print("\U0001F4CA === PARALLEL BATCH SUMMARY: 5 SIMULTANEOUS H100 REQUESTS (Q4-Q8) ===\n")

par_total_in, par_total_out, par_total_calls, par_total_api_calls = 0, 0, 0, 0
individual_durations = []
for label, prompt in parallel_questions:
    outcome = parallel_results.get(label, {})
    m = parallel_metrics.get(label, {})
    if m.get("error") or outcome.get("elapsed") is None:
        print(f"{label}: unavailable")
        continue
    in_tok = m.get("input_tokens") or 0
    out_tok = m.get("output_tokens") or 0
    calls = m.get("tool_call_count") or 0
    api_calls = m.get("api_call_count") or 0
    dur = outcome["elapsed"]
    individual_durations.append(dur)
    par_total_in += in_tok
    par_total_out += out_tok
    par_total_calls += calls
    par_total_api_calls += api_calls
    print(f"{label:38s} | {dur:6.1f}s | tool calls: {calls:3d} | LLM calls: {api_calls:2d} | in: {in_tok:6d} | out: {out_tok:6d}")

print(f"\n{'PARALLEL TOTAL':38s} |        | tool calls: {par_total_calls:3d} | LLM calls: {par_total_api_calls:2d} | in: {par_total_in:6d} | out: {par_total_out:6d}")
print(f"Grand total tokens (parallel batch): {par_total_in + par_total_out}")

if individual_durations:
    sum_sequential_estimate = sum(individual_durations)
    speedup = sum_sequential_estimate / batch_elapsed if batch_elapsed > 0 else float("nan")
    print(f"\n\u23f1  Wall-clock for all {len(individual_durations)} requests IN PARALLEL:                {batch_elapsed:6.1f}s")
    print(f"\u23f1  Sum of each request's own duration (sequential-equivalent): {sum_sequential_estimate:6.1f}s")
    print(f"\u26a1 Concurrency speedup: {speedup:.2f}x")
    print(f"\nA speedup near {len(individual_durations)}x would mean the GPU served all requests")
    print(f"truly in parallel. A speedup near 1x would mean vLLM (or Hermes) is")
    print(f"effectively queueing them one at a time despite the concurrent dispatch.")

📊 === PARALLEL BATCH SUMMARY: 5 SIMULTANEOUS H100 REQUESTS (Q4-Q8) ===

Q4 Creatine & Muscle Strength          |   78.5s | tool calls:   2 | LLM calls:  2 | in:  39662 | out:   1308
Q5 Vitamin D & Respiratory Infections  |   75.5s | tool calls:   2 | LLM calls:  2 | in:  57127 | out:   1153
Q6 Meditation & Cortisol               |   75.3s | tool calls:   2 | LLM calls:  2 | in:  47169 | out:   1136
Q7 HIIT vs Steady-State Cardio         |   79.0s | tool calls:   2 | LLM calls:  2 | in:  43501 | out:   1328
Q8 Probiotics & IBS                    |   80.0s | tool calls:   2 | LLM calls:  2 | in:  44203 | out:   1380

PARALLEL TOTAL                         |        | tool calls:  10 | LLM calls: 10 | in: 231662 | out:   6305
Grand total tokens (parallel batch): 237967

⏱  Wall-clock for all 5 requests IN PARALLEL:                  80.1s
⏱  Sum of each request's own duration (sequential-equivalent):  388.4s
⚡ Concurrency speedup: 4.85x

A speedup near 5x would mean the GPU served all reque

In [47]:
print("\U0001F4CA === GRAND TOTAL: ALL 8 QUESTIONS (3 sequential + 5 parallel) ===\n")

grand_in = seq_total_in + par_total_in
grand_out = seq_total_out + par_total_out
grand_calls = seq_total_calls + par_total_calls
grand_api_calls = seq_total_api_calls + par_total_api_calls

print(f"Sequential (Q1-Q3):  tool calls: {seq_total_calls:3d} | LLM calls: {seq_total_api_calls:2d} | input tok: {seq_total_in:7d} | output tok: {seq_total_out:6d}")
print(f"Parallel   (Q4-Q8):  tool calls: {par_total_calls:3d} | LLM calls: {par_total_api_calls:2d} | input tok: {par_total_in:7d} | output tok: {par_total_out:6d}")
print(f"{'-' * 90}")
print(f"GRAND TOTAL (8 Qs):  tool calls: {grand_calls:3d} | LLM calls: {grand_api_calls:2d} | input tok: {grand_in:7d} | output tok: {grand_out:6d}")
print(f"Grand total tokens across all 8 research questions: {grand_in + grand_out}")

📊 === GRAND TOTAL: ALL 8 QUESTIONS (3 sequential + 5 parallel) ===

Sequential (Q1-Q3):  tool calls:   7 | LLM calls:  6 | input tok:  184611 | output tok:   4115
Parallel   (Q4-Q8):  tool calls:  10 | LLM calls: 10 | input tok:  231662 | output tok:   6305
------------------------------------------------------------------------------------------
GRAND TOTAL (8 Qs):  tool calls:  17 | LLM calls: 16 | input tok:  416273 | output tok:  10420
Grand total tokens across all 8 research questions: 426693


---
## Part 6 — Recap: What Changed in This Notebook

### 🖥️ NOTEBOOK 1 — Local laptop / single L4 GPU

**Limitations**

- 1B laptop model: unreliable or absent tool-calling.
- 8B L4 GPU model: reliable tool-calling, but limited to a moderate-complexity single skill (spam detection) with no external tools.
- No connection to real-world data — answers came from the model's training data only.

### ☁️ NOTEBOOK 2 — Qwen2.5-32B-Instruct on your H100 (this notebook)

**What's new**

- **A genuinely bigger model** — 32B parameters, whose FP8 weights alone (~32GB) already exceed the L4's entire 24GB card. This is hardware the L4 physically cannot run, not just a convenience upgrade.
- **Double the context window** — 128K tokens via YaRN scaling, vs. the 64K floor used elsewhere in this workshop.
- **A real external tool**: Hermes now calls `search_papers` against live academic databases (arXiv, PubMed, bioRxiv, Semantic Scholar, CORE, OpenAlex, Crossref) instead of guessing from memory.
- **Genuine multi-step agentic behavior**: the `/deep-research` skill forces decomposition into sub-questions, multiple tool calls, and citation-backed synthesis — not a single-shot answer.

---

> **Context:** Two other larger models (Hermes-4.3-36B, Gemma 4 31B) were tried for this notebook and both hit real problems — an unresolved load failure and a documented open tool-calling bug, respectively. Qwen2.5-32B-Instruct's own wrinkle (a 32K default context) was a known, documented fix rather than a mystery bug, which is why it was worth the extra setup step over the alternatives. The lesson: a bigger model is only a good demo choice when its rough edges are well-understood, not when they're unknown.

---
## Summary

### Workshop 1 (Notebook 1) — Free/Local + L4 GPU
- Installed Hermes Agent from scratch.
- Ran a 1B model locally with Ollama, then Hermes-3-Llama-3.1-8B on an L4 GPU.
- Learned the Skills format (SKILL.md files).
- Built a Gmail Spam Detector skill.

### Workshop 2 (This Notebook) — H100 + Real Research Tools
- Connected Hermes to Qwen2.5-32B-Instruct on your own H100 VM — a model too large for the L4, running with a YaRN-extended 128K context window.
- Installed and registered `paper-search-mcp` — a free, key-free MCP server for real academic paper search.
- Built a Deep Research skill that does genuine multi-step, tool-calling, citation-backed research.

### The Big Takeaway

**Skills are model-agnostic, and MCP tools are what make "agentic" real — but model choice still matters for what a bigger GPU actually buys you.** A larger, well-established model with a documented context-extension path gives you real new capability (bigger model, longer context) that a smaller GPU cannot offer, while still relying on the same MCP tool connection that turns Hermes from "a chatbot that might be right" into "an agent that checks its work against real sources."

```
# Notebook 1 (L4):
model: custom/hermes-3-llama-3.1-8b

# Notebook 2 (H100) - bigger model, longer context, plus a real tool:
model: custom/Qwen2.5-32B-Instruct
mcp_servers:
  paper-search: { command: python, args: [-m, paper_search_mcp.server] }
```

---

### Resources
- **Hermes Agent**: https://hermes-agent.nousresearch.com
- **Hermes Skills Hub**: https://agentskills.io
- **Qwen2.5-32B-Instruct model card**: https://huggingface.co/Qwen/Qwen2.5-32B-Instruct
- **Qwen2.5 YaRN long-context guide**: https://huggingface.co/Qwen/Qwen2.5-32B-Instruct/discussions/5
- **paper-search-mcp**: https://github.com/openags/paper-search-mcp
- **Consensus MCP docs** (optional bonus): https://docs.consensus.app/docs/mcp
- **vLLM docs**: https://docs.vllm.ai
- **Hermes GitHub**: https://github.com/NousResearch/hermes-agent